# Instructblip Flan T5 Xl COCO Baseline

This notebook was reorganized for the GitHub reproducibility package.
Original file: `COCO-Baseline/InstructBLIP_BaseModel`.

**Security note:** hard-coded Hugging Face tokens were removed. Use interactive login or environment variables instead.


No:t InstructBLIP ilk deneme notebookunda var, silmeyelim.


1- https://storage.googleapis.com/sfr-vision-language-research/datasets/coco_karpathy_test_gt.json --> ben de test_gt.json olarak bu dosyayı kullanıyorum, metrik oluştururken burası baz alınacak

2-https://storage.googleapis.com/sfr-vision-language-research/datasets/coco_karpathy_test.json --> preds üretmek için ben de buradaki dosyayı kullanıyorum

özetle; orjinal makaledeki gibi yapıyoruz. Drivedaki dosyaları indirip bu bağlantılardaki dosyalar ile istersek kıyaslayalım aynı şeyler olduğunu göreceğiz.

# **Drive Bağlantısı ve Dosya Kontrolleri**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p "/content/drive/MyDrive/datasets/coco2014"
%cd "/content/drive/MyDrive/datasets/coco2014"
!pwd
!ls -la


In [ ]:
import json
p = "/content/drive/MyDrive/datasets/coco2014/karpathy/coco_karpathy_test.json"
with open(p,"r") as f:
    x = json.load(f)
print(type(x))
print("len:", len(x) if isinstance(x, list) else "keys:"+str(list(x.keys())[:10]))
print("sample:", x[0] if isinstance(x, list) else {k:type(v) for k,v in list(x.items())[:3]})


In [ ]:
# Drive ana kök
DRIVE_ROOT = "/content/drive/MyDrive"

# COCO 2014 ana dizin
COCO_ROOT = f"{DRIVE_ROOT}/datasets/coco2014"

# Görseller (VAL)
IMG_DIR = f"{COCO_ROOT}/val2014"

# COCO orijinal annotation (gerekirse)
ANN_DIR  = f"{COCO_ROOT}/annotations"
ANN_PATH = f"{ANN_DIR}/captions_val2014.json"

# Karpathy split dizini
KARPATHY_DIR = f"{COCO_ROOT}/karpathy"

KARPATHY_TRAIN = f"{KARPATHY_DIR}/coco_karpathy_train.json"
KARPATHY_VAL   = f"{KARPATHY_DIR}/coco_karpathy_val.json"
KARPATHY_TEST  = f"{KARPATHY_DIR}/coco_karpathy_test.json"

# (Opsiyonel) Birleşik dataset dosyası – artık gerekmez ama dursun
DATASET_COCO = f"{KARPATHY_DIR}/dataset_coco.json"


In [ ]:
import os, json
from PIL import Image

print("IMG_DIR exists:", os.path.isdir(IMG_DIR))
print("ANN_PATH exists:", os.path.exists(ANN_PATH))
print("KARPATHY_TEST exists:", os.path.exists(KARPATHY_TEST))

with open(KARPATHY_TEST, "r") as f:
    items = json.load(f)

print("Karpathy test len:", len(items))
print("Sample item:", items[0])

# Görsel açma testi
img_rel = items[0]["image"]              # 'val2014/COCO_val2014_....jpg'
img_path = os.path.join(COCO_ROOT, img_rel)

print("Resolved image path:", img_path)
print("Image exists:", os.path.exists(img_path))

img = Image.open(img_path).convert("RGB")
print("Image size:", img.size)


In [ ]:
# json dosyasındaki id'ler parse ediliyor

import re

def coco_id_from_relpath(rel_path: str) -> int:
    # rel_path örn: "val2014/COCO_val2014_000000391895.jpg"
    m = re.search(r"_(\d{12})\.jpg$", rel_path)
    if not m:
        raise ValueError(f"COCO id parse edilemedi: {rel_path}")
    return int(m.group(1))  # "000000391895" -> 391895


# **InstructBLIP Model Yükleme --> 4.023 B**

In [ ]:
import torch
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration

MODEL_ID = "Salesforce/instructblip-flan-t5-xl"

# LAVIS config'ten birebir, orjinal githup reposu
PROMPT = "A short image caption."
NUM_BEAMS = 5
MAX_NEW_TOKENS = 80   # LAVIS: max_len
MIN_NEW_TOKENS = 10   # LAVIS: min_len
LENGTH_PENALTY = 1.0
DO_SAMPLE = False

# image_size (int, optional, defaults to 224) — The size (resolution) of each image.
processor = InstructBlipProcessor.from_pretrained(MODEL_ID)

model = InstructBlipForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,   # LAVIS amp: True
).eval()

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

def iblip_generate(img):
    inputs = processor(images=img, text=PROMPT, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            do_sample=DO_SAMPLE,
            num_beams=NUM_BEAMS,
            max_new_tokens=MAX_NEW_TOKENS,
            min_new_tokens=MIN_NEW_TOKENS,
            length_penalty=LENGTH_PENALTY,
            early_stopping=True,   # beam search'te iyi pratik
        )

    cap = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    # LAVIS "blip_caption" genelde agresif temizlemez; burada minimal tut
    cap = cap.strip().replace("\n", " ")
    return cap


# **Test - Inference ve Sonuçların Kayıt Altına Alınması**

In [ ]:
preds = []
seen = set()


In [ ]:
for it in items:
    rel = it["image"]   # "val2014/COCO_val2014_000000391895.jpg"

    image_id = coco_id_from_relpath(rel)   # <-- ÇÖZÜMÜN KOYULACAĞI SATIR (tam burası)

    img_path = os.path.join(COCO_ROOT, rel)
    img = Image.open(img_path).convert("RGB")

    cap = iblip_generate(img)

    preds.append({"image_id": image_id, "caption": cap})


In [ ]:
preds[:3]


In [ ]:
len(preds)

In [ ]:
import json, os, shutil
from google.colab import drive

# 🔒 Güvenlik kontrolü
assert "preds" in globals(), "❌ preds değişkeni yok. Önce inference çalışmalı."
assert isinstance(preds, list) and len(preds) > 0, "❌ preds boş. Inference tamamlanmamış."

# Drive mount
drive.mount('/content/drive')

# Geçici eval yolu (eval kodların burayı kullanıyor)
TMP_DIR = "/content/eval"
os.makedirs(TMP_DIR, exist_ok=True)
tmp_pred_path = f"{TMP_DIR}/instructblip_preds.json"

# Kalıcı Drive yolu
PERM_DIR = "/content/drive/MyDrive/vlm_eval"
os.makedirs(PERM_DIR, exist_ok=True)
perm_pred_path = f"{PERM_DIR}/instructblip_preds.json"

# Yaz → kopyala
with open(tmp_pred_path, "w") as f:
    json.dump(preds, f)

shutil.copy(tmp_pred_path, perm_pred_path)

# Log
print("✅ preds başarıyla kaydedildi")
print("   Temporary :", tmp_pred_path)
print("   Permanent :", perm_pred_path)
print("   num_preds :", len(preds))
print("   sample    :", preds[0])


# **Başarı Metriklerinin Hazırlanması ve Sonuçlar**

In [ ]:
# metriklerin kurulumu

!pip -q install pycocotools
!pip -q install git+https://github.com/salaniz/pycocoevalcap
!apt-get -qq update
!apt-get -qq install -y default-jre


In [ ]:
# SPICE Metriğini Çıkarıyoruz, sorun --> ARIZA!

import inspect, pycocoevalcap
from pathlib import Path
import pycocoevalcap.eval as pe

eval_path = Path(inspect.getfile(pe))
print("eval.py path:", eval_path)

txt = eval_path.read_text()

# SPICE scorer satırını kaldır (varsa)
txt2 = txt.replace("from .spice.spice import Spice\n", "")
txt2 = txt2.replace("(Spice(), \"SPICE\")", "")

eval_path.write_text(txt2)
print("Patched eval.py (SPICE removed). Restart runtime önerilir ama çoğu zaman gerekmez.")


In [ ]:
from pycocotools.coco import COCO
from pycocoevalcap.eval import COCOEvalCap

gt_path = "/content/drive/MyDrive/coco_karpathy/coco_karpathy_test_gt.json"
pred_path = "/content/eval/instructblip_preds.json"

coco = COCO(gt_path)
cocoRes = coco.loadRes(pred_path)

cocoEval = COCOEvalCap(coco, cocoRes)
cocoEval.params["image_id"] = coco.getImgIds()   # 5000 id
cocoEval.evaluate()

metrics = cocoEval.eval
metrics


In [ ]:
print("=== InstructBLIP baseline | COCO Karpathy test (5k) ===")
for k in ["CIDEr", "Bleu_4", "METEOR", "ROUGE_L"]:
    print(f"{k:8s}: {metrics.get(k, None)}")


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"Total parameters (B): {total_params / 1e9:.3f} B")


# **InternVL Model Yükleme --> 4B**